# Fraud Detection - Notebook 5: Drift Detection and Retraining

### What this notebook covers
1. **ADWIN** (ADaptive WINdowing) — detects distributional drift in the ensemble's predicted fraud rate by maintaining an adaptive window whose size shrinks when drift is detected
2. **Page-Hinkley** — detects gradual concept drift in per-transaction reconstruction error (a slower, more sensitive test than ADWIN)
3. **MLflow** — logs every retraining run with parameters, metrics, and artefact paths so experiments are reproducible and comparable
4. **Retraining trigger** — when either detector fires, the pipeline retrains LightGBM on a sliding window of recent data and logs the updated model

### Drift detection philosophy
- **ADWIN** is best for sudden drift: e.g. a new fraud ring starts operating and the predicted score distribution shifts abruptly
- **Page-Hinkley** is best for gradual drift: e.g. normal spending patterns slowly change over months, eroding model performance
- Both detectors are run in parallel; either firing triggers a retraining check

### Operating threshold
The cost-optimal threshold from notebook 4 (0.478) is used throughout — not the F1 threshold.

In [3]:
import sys, json, numpy as np, pandas as pd, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

IEEE_DIR  = Path('/kaggle/input/competitions/ieee-fraud-detection')
SRC_DIR   = Path('/kaggle/input/datasets/youssefmousaaid/fraud-detection-src')
WORK_DIR  = Path('/kaggle/working')
PROC_DIR  = WORK_DIR / 'processed'
MDL_DIR   = WORK_DIR / 'models'
OUT_DIR   = WORK_DIR / 'outputs'
MLRUNS    = WORK_DIR / 'mlruns'

for d in [PROC_DIR, MDL_DIR, OUT_DIR, MLRUNS,
          OUT_DIR/'drift', OUT_DIR/'pipeline']:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

for pkg in ['mlflow', 'river']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Packages ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 59.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


Packages ready


In [4]:
import pickle, gc, time, mlflow
import lightgbm as lgb
from river import drift as river_drift
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'#0F1117','axes.facecolor':'#1A1D27',
    'axes.edgecolor':'#2E3347','text.color':'#E0E0E0','axes.labelcolor':'#E0E0E0',
    'xtick.color':'#9AA0B0','ytick.color':'#9AA0B0','grid.color':'#2E3347',
    'grid.linestyle':'--','font.family':'monospace'})
PAL = {'drift':'#E84545','normal':'#4A90D9','retrain':'#F5A623','lgb':'#A259FF'}

mlflow.set_tracking_uri(str(MLRUNS))
mlflow.set_experiment('fraud_detection')
print(f'MLflow tracking URI: {MLRUNS}')
print(f'river version      : {river_drift.ADWIN.__module__.split(".")[0]}')

AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [ ]:
# ── Load data and models ────────────────────────────────────────────────
with open(PROC_DIR/'graph_meta.json') as f:
    GRAPH_FEATURES = json.load(f)['graph_features']
with open(PROC_DIR/'sequence_meta.json') as f:
    meta = json.load(f)
N_TAB        = meta['n_features']
FEATURE_NAMES = meta['features'] + GRAPH_FEATURES

with open(MDL_DIR/'ensemble_results.json') as f:
    results = json.load(f)
OPERATING_THR = results['threshold_cost']   # 0.478 — cost-optimal from notebook 4
print(f'Operating threshold (cost-optimal): {OPERATING_THR:.4f}')

# Load aligned test features and scores
xgb_scores   = np.load(PROC_DIR/'test_scores.npy')
lgb_scores   = np.load(PROC_DIR/'lgb_scores_test.npy')
G_test       = np.load(PROC_DIR/'graph_features_test.npy')
g_tx_test    = np.load(PROC_DIR/'graph_tx_ids_test.npy')
y_test       = np.load(PROC_DIR/'y_graph_test.npy')
X_test_tab   = np.load(PROC_DIR/'X_test_tab.npy')
tx_ids_test  = np.load(PROC_DIR/'tx_ids_test_tab.npy')

# Reconstruct ensemble probabilities using saved meta-learner
with open(MDL_DIR/'ensemble.pkl','rb') as f:
    ens = pickle.load(f)
meta_learner  = ens['meta_learner']
ss_xgb        = ens['score_scaler_xgb']
ss_lgb        = ens['score_scaler_lgb']

def safe_norm(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-9)

xn = safe_norm(xgb_scores)
ln = safe_norm(lgb_scores)
X_meta     = np.column_stack([xn, ln, xn*ln, np.abs(xn-ln), np.maximum(xn,ln)])
ens_proba  = meta_learner.predict_proba(X_meta)[:,1]

# Join tabular features to graph test rows for retraining
tab_lookup = {int(tid): i for i, tid in enumerate(tx_ids_test)}
X_tab_block = np.zeros((len(G_test), N_TAB), dtype=np.float32)
for i, tid in enumerate(g_tx_test):
    idx = tab_lookup.get(int(tid))
    if idx is not None:
        X_tab_block[i] = X_test_tab[idx]
X_test_full = np.hstack([X_tab_block, G_test])
del X_tab_block, X_test_tab; gc.collect()

# Load training data for retraining
G_train          = np.load(PROC_DIR/'graph_features_train.npy')
g_tx_train       = np.load(PROC_DIR/'graph_tx_ids_train.npy')
y_train          = np.load(PROC_DIR/'y_graph_train.npy')
X_train_tab_all  = np.load(PROC_DIR/'X_train_tab_all.npy')
tx_ids_train     = np.load(PROC_DIR/'tx_ids_train_all.npy')

tab_train_lookup = {int(tid): i for i, tid in enumerate(tx_ids_train)}
X_tab_train = np.zeros((len(G_train), N_TAB), dtype=np.float32)
for i, tid in enumerate(g_tx_train):
    idx = tab_train_lookup.get(int(tid))
    if idx is not None:
        X_tab_train[i] = X_train_tab_all[idx]
X_train_full = np.hstack([X_tab_train, G_train])
del X_tab_train, X_train_tab_all, G_train; gc.collect()

print(f'Ensemble proba range: [{ens_proba.min():.4f}, {ens_proba.max():.4f}]')
print(f'X_train: {X_train_full.shape} | X_test: {X_test_full.shape}')

In [ ]:
# ── Simulate a production stream ────────────────────────────────────────
# In production, transactions arrive one-by-one in real time.
# We simulate this by replaying the test set in temporal order,
# feeding each transaction's ensemble score to both drift detectors.
#
# To make drift detection meaningful we inject a synthetic concept drift
# event at the 60% mark: fraud transactions in the latter 40% have their
# scores artificially reduced by 0.3 (simulating a new fraud pattern the
# model hasn't seen). This causes the predicted fraud rate to drop while
# true fraud continues — a realistic drift scenario.

DRIFT_INJECT_FRAC = 0.60    # drift starts at 60% through the stream
DRIFT_MAGNITUDE   = 0.30    # score reduction applied to fraud rows post-drift

n_stream  = len(ens_proba)
drift_start = int(n_stream * DRIFT_INJECT_FRAC)

# Build drifted score stream
stream_scores = ens_proba.copy()
fraud_mask    = (y_test == 1)
post_drift    = np.arange(n_stream) >= drift_start
drifted_mask  = fraud_mask & post_drift
stream_scores[drifted_mask] = np.clip(
    stream_scores[drifted_mask] - DRIFT_MAGNITUDE, 0, 1
)

print(f'Stream length   : {n_stream:,}')
print(f'Drift injected  : position {drift_start:,} ({DRIFT_INJECT_FRAC*100:.0f}%)')
print(f'Rows affected   : {drifted_mask.sum():,} fraud transactions')
print(f'Pre-drift fraud rate in top alerts : '
      f'{(stream_scores[:drift_start] >= OPERATING_THR).mean()*100:.2f}%')
print(f'Post-drift fraud rate in top alerts: '
      f'{(stream_scores[drift_start:] >= OPERATING_THR).mean()*100:.2f}%')

In [ ]:
# ── ADWIN detector ──────────────────────────────────────────────────────
# ADWIN maintains an adaptive window over a binary stream.
# Here the stream is: 1 if the transaction was flagged as fraud
# (score >= threshold), 0 otherwise.
# When the mean of the window changes significantly, ADWIN fires.
#
# delta: confidence parameter. Smaller = more sensitive (more false alarms).
# 0.002 is the standard default; use 0.05 for noisier streams.

adwin = river_drift.ADWIN(delta=0.002)
adwin_detections = []   # positions where ADWIN fired
adwin_means      = []   # running mean at each position (for plotting)

for i, score in enumerate(stream_scores):
    flagged = int(score >= OPERATING_THR)
    adwin.update(flagged)
    adwin_means.append(adwin.estimation)
    if adwin.drift_detected:
        adwin_detections.append(i)
        adwin = river_drift.ADWIN(delta=0.002)   # reset after detection

print(f'ADWIN detections: {len(adwin_detections)}')
if adwin_detections:
    for pos in adwin_detections:
        lag = pos - drift_start
        print(f'  Position {pos:,}  (lag from injection: {lag:+,})')

In [ ]:
# ── Page-Hinkley detector ───────────────────────────────────────────────
# Page-Hinkley detects a persistent upward shift in a continuous signal.
# Here the signal is the raw ensemble score — we detect when the mean
# score drops (indicating the model is becoming less confident on fraud).
# PH is run on the NEGATED score so a drop becomes a detectable rise.
#
# min_instances: burn-in period before detection starts
# delta        : minimum magnitude of change to detect
# threshold    : cumulative sum threshold before firing
# alpha        : forgetting factor (1.0 = no forgetting)

ph = river_drift.PageHinkley(min_instances=30, delta=0.005,
                              threshold=50, alpha=0.9999)
ph_detections = []
ph_cumsum     = []

for i, score in enumerate(stream_scores):
    ph.update(-score)   # negate: drop in score = rise in -score
    ph_cumsum.append(getattr(ph, '_sum', 0.0))
    if ph.drift_detected:
        ph_detections.append(i)
        ph = river_drift.PageHinkley(min_instances=30, delta=0.005,
                                      threshold=50, alpha=0.9999)

print(f'Page-Hinkley detections: {len(ph_detections)}')
if ph_detections:
    for pos in ph_detections:
        lag = pos - drift_start
        print(f'  Position {pos:,}  (lag from injection: {lag:+,})')

In [ ]:
# ── Drift visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
fig.suptitle('Drift Detection — Simulated Production Stream',
             fontsize=14, fontweight='bold', color='#E0E0E0')

# Rolling alert rate (100-transaction window)
window     = 100
alert_rate = pd.Series((stream_scores >= OPERATING_THR).astype(float))\
               .rolling(window, min_periods=1).mean().values

axes[0].plot(alert_rate, color=PAL['normal'], lw=1.2, label='Alert rate (rolling 100)')
axes[0].axvline(drift_start, color=PAL['drift'], lw=2, ls='--', label='Drift injected')
for pos in adwin_detections:
    axes[0].axvline(pos, color=PAL['retrain'], lw=1.5, ls=':',
                    label='ADWIN detection' if pos == adwin_detections[0] else '')
for pos in ph_detections:
    axes[0].axvline(pos, color=PAL['lgb'], lw=1.5, ls='-.',
                    label='Page-Hinkley detection' if pos == ph_detections[0] else '')
axes[0].set_ylabel('Alert rate')
axes[0].set_title('Rolling alert rate with drift detections', color='#E0E0E0')
axes[0].legend(fontsize=9)

# ADWIN window mean
axes[1].plot(adwin_means, color=PAL['normal'], lw=1.2, label='ADWIN window mean')
axes[1].axvline(drift_start, color=PAL['drift'], lw=2, ls='--', label='Drift injected')
for pos in adwin_detections:
    axes[1].axvline(pos, color=PAL['retrain'], lw=1.5, ls=':',
                    label='ADWIN detection' if pos == adwin_detections[0] else '')
axes[1].set_xlabel('Transaction index (stream order)')
axes[1].set_ylabel('ADWIN mean')
axes[1].set_title('ADWIN adaptive window mean', color='#E0E0E0')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR/'drift/drift_detection.png', dpi=150,
            bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# ── Retraining pipeline with MLflow ────────────────────────────────────
# When drift is detected, we retrain LightGBM on a sliding window of
# recent data. In production this window would be the last N days of
# labelled transactions. Here we simulate it by training on the most
# recent RETRAIN_FRAC fraction of the training set.
#
# MLflow logs: parameters, pre/post metrics, model artefact path.

RETRAIN_FRAC = 0.30   # use most recent 30% of training data

def evaluate_lgb(model, X, y, thr):
    proba = model.predict_proba(X)[:,1]
    auroc = roc_auc_score(y, proba)
    auprc = average_precision_score(y, proba)
    yp    = (proba >= thr).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y, yp, labels=[0,1]).ravel()
    return {'auroc': auroc, 'auprc': auprc,
            'precision': tp_/(tp_+fp_+1e-9),
            'recall'   : tp_/(tp_+fn_+1e-9),
            'fn': int(fn_), 'fp': int(fp_)}

# Baseline metrics (model from notebook 4)
baseline = evaluate_lgb(ens['lgb_model'], X_test_full, y_test, OPERATING_THR)
print('Baseline LightGBM metrics:')
for k, v in baseline.items():
    print(f'  {k:12s}: {v:.4f}' if isinstance(v, float) else f'  {k:12s}: {v}')

# Sliding window: most recent RETRAIN_FRAC of training rows
n_retrain  = int(len(X_train_full) * RETRAIN_FRAC)
X_recent   = X_train_full[-n_retrain:]
y_recent   = y_train[-n_retrain:]
print(f'\nRetrain window: {n_retrain:,} rows | fraud={y_recent.sum()} ({y_recent.mean()*100:.2f}%)')

if y_recent.sum() == 0:
    print('WARNING: no fraud in retrain window — skipping retraining')
else:
    n_norm = int((y_recent == 0).sum())
    n_fr   = int((y_recent == 1).sum())
    spw    = n_norm / max(n_fr, 1)

    retrain_params = {
        'objective'        : 'binary',
        'metric'           : 'average_precision',
        'n_estimators'     : 500,
        'learning_rate'    : 0.05,
        'num_leaves'       : 63,
        'min_child_samples': 20,
        'subsample'        : 0.8,
        'subsample_freq'   : 1,
        'colsample_bytree' : 0.8,
        'reg_alpha'        : 0.1,
        'reg_lambda'       : 1.0,
        'scale_pos_weight' : spw,
        'random_state'     : 42,
        'device'           : 'gpu',
        'verbose'          : -1,
    }

    with mlflow.start_run(run_name='retrain_drift_trigger') as run:
        mlflow.log_params({
            'retrain_frac'       : RETRAIN_FRAC,
            'retrain_n_rows'     : n_retrain,
            'drift_position_adwin': adwin_detections[0] if adwin_detections else -1,
            'drift_position_ph'  : ph_detections[0]    if ph_detections    else -1,
            'operating_threshold': OPERATING_THR,
            **{f'lgb_{k}': v for k, v in retrain_params.items()
               if k not in ('verbose','device')},
        })

        # Log baseline
        mlflow.log_metrics({f'baseline_{k}': v for k, v in baseline.items()
                            if isinstance(v, float)})

        # Retrain
        t0 = time.time()
        retrained_lgb = lgb.LGBMClassifier(**retrain_params, n_jobs=-1)

        # Use a validation split for early stopping
        X_rt, X_rv, y_rt, y_rv = train_test_split(
            X_recent, y_recent, test_size=0.2,
            random_state=42, stratify=y_recent
        )
        retrained_lgb.fit(
            X_rt, y_rt,
            eval_set        = [(X_rv, y_rv)],
            callbacks       = [lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)],
        )
        train_time = time.time() - t0
        print(f'Retrained in {train_time:.1f}s | best_iteration={retrained_lgb.best_iteration_}')

        # Evaluate retrained model
        retrained_metrics = evaluate_lgb(retrained_lgb, X_test_full, y_test, OPERATING_THR)
        mlflow.log_metrics({f'retrained_{k}': v for k, v in retrained_metrics.items()
                            if isinstance(v, float)})
        mlflow.log_metric('train_time_s', train_time)

        # Save and log model
        retrain_path = MDL_DIR / 'lgb_retrained.pkl'
        with open(retrain_path, 'wb') as f: pickle.dump(retrained_lgb, f)
        mlflow.log_artifact(str(retrain_path))

        run_id = run.info.run_id

    print(f'\nRetrained LightGBM metrics:')
    for k, v in retrained_metrics.items():
        delta = (v - baseline[k]) if isinstance(v, float) else ''
        delta_str = f' ({delta:+.4f})' if isinstance(delta, float) else ''
        print(f'  {k:12s}: {v:.4f}{delta_str}' if isinstance(v, float)
              else f'  {k:12s}: {v}')
    print(f'\nMLflow run ID: {run_id}')

In [ ]:
# ── MLflow run summary ──────────────────────────────────────────────────
print('MLflow experiment runs:')
client = mlflow.tracking.MlflowClient(tracking_uri=str(MLRUNS))
experiment = client.get_experiment_by_name('fraud_detection')
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['start_time DESC'],
    max_results=5,
)
for r in runs:
    auroc_b = r.data.metrics.get('baseline_auroc', float('nan'))
    auroc_r = r.data.metrics.get('retrained_auroc', float('nan'))
    print(f"  Run: {r.info.run_name:<35} | "
          f"baseline AUROC={auroc_b:.4f} | retrained AUROC={auroc_r:.4f}")

In [ ]:
# ── Pipeline summary and drift report ──────────────────────────────────
report = {
    'drift_injection_position' : drift_start,
    'drift_injection_fraction' : DRIFT_INJECT_FRAC,
    'drift_magnitude'          : DRIFT_MAGNITUDE,
    'adwin_detections'         : adwin_detections,
    'ph_detections'            : ph_detections,
    'adwin_lag'  : (adwin_detections[0] - drift_start) if adwin_detections else None,
    'ph_lag'     : (ph_detections[0]    - drift_start) if ph_detections    else None,
    'operating_threshold'      : OPERATING_THR,
    'baseline_auroc'           : round(baseline['auroc'], 4),
    'retrained_auroc'          : round(retrained_metrics['auroc'], 4)
                                 if y_recent.sum() > 0 else None,
    'mlflow_run_id'            : run_id if y_recent.sum() > 0 else None,
}
with open(OUT_DIR/'drift/drift_report.json','w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))